In [0]:
from pyspark.sql import functions as f
from datetime import datetime
from delta.tables import DeltaTable
import uuid
from pyspark.sql.window import Window

In [0]:
spark.sql("use catalog ecommerce_data")
spark.sql("create schema if not exists silver")

silver_run_id=str(uuid.uuid4())

print("Silver run id:",silver_run_id)

In [0]:
spark.sql("""
          create table if not exists ecommerce_data.silver.silver_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_ingested_at timestamp,
              rows_merged bigint,
              silver_run_id string,
              run_status string,
              updated_at timestamp
          )
          using delta
          """)

1st helper function
upsert_to_silver() -> if table exists update records 
else create new table

In [0]:
def upsert_to_silver(source_table,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        df=DeltaTable.forName(spark,target_table)
        (
            df.alias('target')
            .merge(source_table.alias('source'),f"target.{join_key}=source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        source_table.write.format('delta').mode('append').saveAsTable(target_table) 


2nd helper function

gets the latest ingested at timestamp processed by silver from bronze

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
        lp=(spark.table("ecommerce_data.silver.silver_control").filter(
                  (f.col('layer')=='silver') &
                  (f.col('entity_name')==entity_name) &
                  (f.col('run_status')=='success'))
              .orderBy(f.col('updated_at').desc())
              .limit(1))

        rows=lp.collect()

        if not rows:
             return None   
        return rows[0]['last_processed_ingested_at']


3rd helper function 
update silver control table after successfull silver load

In [0]:
def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_ingested_at,
    rows_merged
):
    control_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                str(last_processed_bronze_run_id),
                last_processed_ingested_at,
                int(rows_merged),
                str(silver_run_id),
                "success",
                datetime.utcnow()
            )
        ],
        schema="""
            layer STRING,
            entity_name STRING,
            last_processed_bronze_run_id STRING,
            last_processed_ingested_at TIMESTAMP,
            rows_merged INT,
            silver_run_id STRING,
            run_status STRING,
            updated_at TIMESTAMP
        """
    )

    target_table = DeltaTable.forName(
        spark,
        "ecommerce_data.silver.silver_control"
    )

    (
        target_table.alias("target")
        .merge(
            control_df.alias("source"),
            """
            target.entity_name = source.entity_name
            AND target.layer = source.layer
            """
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id":
                    "source.last_processed_bronze_run_id",
                "last_processed_ingested_at":
                    "source.last_processed_ingested_at",
                "rows_merged": "source.rows_merged",
                "silver_run_id": "source.silver_run_id",
                "run_status": "source.run_status",
                "updated_at": "source.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

4th helper function
get new data from bronze

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_updated_at=get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)
    if last_updated_at is None:
        return bronze_df,last_updated_at
    return bronze_df.filter(f.col('bronze_ingested_at')>f.lit(last_updated_at)),last_updated_at

Orders Table Transformation

In [0]:
silver_df,last_updated_at = get_incremental_bronze('ecommerce_data.bronze.orders_raw','orders')

silver_df_count=silver_df.count()

print("Total number of records in the incremental dataframe: ",silver_df_count)

if silver_df_count > 0:
    window_df=Window.partitionBy('order_id').orderBy(f.col('updated_at').cast('timestamp').desc(),f.col('bronze_ingested_at').desc())
    silver_df_cleaned=(
        silver_df
        .withColumn('order_status',f.upper(f.trim(f.col('order_status'))))
        .withColumn('order_status',f.when(f.col('order_status')=="",None).otherwise(f.col('order_status')))
        .withColumn('order_amount',f.regexp_replace(f.col('order_amount'),r'[$, ]',''))
        .withColumn('order_amount',f.when(f.col('order_amount').isin('','NULL','NA','??','N/A'),None).otherwise(f.col('order_amount')))
        .withColumn('order_amount',f.col('order_amount').cast('double'))
        .withColumn('created_at',f.to_timestamp(f.col('created_at')))
        .withColumn('updated_at',f.to_timestamp(f.col('updated_at')))
        .withColumn('rank',f.row_number().over(window_df))
        .filter(f.col('rank')==1)
        .drop('rank')
        .withColumn('silver_run_id',f.lit(silver_run_id))
    )

    upsert_to_silver(silver_df_cleaned,'ecommerce_data.silver.orders_cleaned','order_id')

    orders_validated=(
        silver_df_cleaned
        .withColumn('to_be_verified_by_orders_team',
        f.when(f.col('order_id').isNull(),'verify order_id')
        .when(f.col('customer_id').isNull(),'verify customer_id')
        .when(f.col('product_id').isNull(),'verify product_id')
        .when(f.col('order_status').isNull() | (f.trim(f.col('order_status'))==""),'verify order_status')
        .when(f.col('order_amount').isNull() | (f.col('order_amount')<=0),'verify order_amount')
        .otherwise('No Issues')       
        )
        .withColumn('check_order_amount',
                    f.when((f.col('order_amount')<=0) | f.col('order_amount').isNull(),f.lit(True))
                    .otherwise(f.lit(False))
                    )
        .withColumn('order_date',f.to_date('created_at'))
        .withColumn('year',f.year('order_date'))
        .withColumn('month',f.month('order_date'))
        .withColumn('day',f.dayofmonth('order_date'))
        .withColumn('dow',f.date_format('created_at','E'))
    )

    orders_good=orders_validated.filter(f.col('to_be_verified_by_orders_team')=='No Issues')

    orders_bad=(
        orders_validated.filter(f.col('to_be_verified_by_orders_team')!='No Issues')
                    .withColumn('quarantine_ts',f.current_timestamp())
    )

    upsert_to_silver(orders_good,'ecommerce_data.silver.orders_transformed','order_id')
    
    orders_bad.write.format('delta').mode('append').saveAsTable('ecommerce_data.silver.orders_quarantine')

    mx_ingested=silver_df.agg(f.max('bronze_ingested_at').alias('mx')).collect()[0]['mx']

    mx_run=(
        silver_df.filter(f.col('bronze_ingested_at')==f.lit(mx_ingested)).agg(f.max('bronze_run_id').alias('mx')).collect()[0]['mx']
    )
    upsert_silver_control('orders',mx_run,mx_ingested,orders_good.count())

else:
    print("No new records to process from Bronze table")    

    upsert_silver_control(
        'orders',
        None,
        last_updated_at,
        silver_df_count
    )

Products Table Transformation 

In [0]:
products_df,last_updated_at = get_incremental_bronze('ecommerce_data.bronze.products_raw','products')

products_df_count=products_df.count()

print("Total number of records in the incremental dataframe: ",products_df_count)

if products_df_count>0:
    window_df=Window.partitionBy('product_id').orderBy(f.col('updated_at').cast('timestamp').desc(),f.col('bronze_ingested_at').desc())

    products_df_cleaned=(
        products_df
        .withColumn('product_name',f.initcap(f.trim(f.col('product_name'))))
        .withColumn('product_name',f.regexp_replace(f.col('product_name'),r'[-_]',' '))
        .withColumn('product_name',f.when(f.col('product_name')=="",None).otherwise(f.col('product_name')))
        .withColumn('category',f.when(f.upper(f.trim(f.col('category'))) == 'ELECTRNICS', 'Electronics').otherwise(f.initcap(f.trim(f.col('category')))))
        .withColumn('price',f.trim(f.col('price')))
        .withColumn('price',f.regexp_replace(f.col('price'),r'\$',''))
        .withColumn('price',f.regexp_replace(f.col('price'),r',','.'))
        .withColumn('price',f.regexp_replace(f.col('price'),r'\s+',''))
        .withColumn('price',f.expr('try_cast(price as double)'))
        .withColumn('updated_at',f.to_timestamp(f.col('updated_at')))
        .withColumn('rank',f.row_number().over(window_df))
        .filter(f.col('rank')==1)
        .drop('rank')
        .withColumn('silver_run_id',f.lit(silver_run_id)))

    upsert_to_silver(products_df_cleaned,'ecommerce_data.silver.products_cleaned','product_id')

    products_validated=(
        products_df_cleaned
        .withColumn('to_be_verified_by_products_team',
        f.when(f.col('product_name').isNull(),'verify product_name')
        .when(f.col('category').isNull(),'verify category')
        .when(f.col('price').isNull() | (f.col('price')<=0),'verify price')
        .otherwise('No Issues')       
        )
        .withColumn('check_product_price',
                    f.when((f.col('price')<=0) | f.col('price').isNull(),'Invalid_Price')
                    .otherwise('Valid_Price'))
    )

    products_good=products_validated.filter((f.col('to_be_verified_by_products_team')=='No Issues') &
    (f.col('check_product_price')=='Valid_Price'))

    products_bad=(
        products_validated.filter((f.col('to_be_verified_by_products_team')!='No Issues') | 
                                  (f.col('check_product_price')=='Invalid_Price'))
                            .withColumn('quarantine_ts',f.current_timestamp())
    )

    upsert_to_silver(products_good,'ecommerce_data.silver.products_transformed','product_id')

    products_bad.write.format('delta').mode('append').saveAsTable('ecommerce_data.silver.products_quarantine')

    mx_ingested=products_df.agg(f.max('bronze_ingested_at').alias('mx')).collect()[0]['mx']

    mx_run=(
        products_df.filter(f.col('bronze_ingested_at')==f.lit(mx_ingested)).agg(f.max('bronze_run_id').alias('mx')).collect()[0]['mx']
    )

    upsert_silver_control('products',mx_run,mx_ingested,products_good.count())
else:
    print("No new records to process from Bronze table")  

    upsert_silver_control('products',None,last_updated_at,products_df_count) 


Payments Table transformation

In [0]:
payments_df,last_payment_processed_at=get_incremental_bronze('ecommerce_data.bronze.payments_raw','payments')

payments_df_count=payments_df.count()

print("Total Number of records in paymnets incremental Dataframe:",payments_df_count)

if payments_df_count>0:
    window_df=Window.partitionBy('payment_id').orderBy(f.col('bronze_ingested_at').desc())

    payments_df_cleaned=(
        payments_df
        .withColumn('payment_status',f.upper(f.trim(f.col('payment_status'))))
        .withColumn('payment_status',f.when(f.col('payment_status')=="",f.lit(None)).otherwise(f.col('payment_status')))
        .withColumn('paid_amount',f.trim(f.col('paid_amount')))
        .withColumn('paid_amount',f.regexp_replace(f.col('paid_amount'),r'\$',''))
        .withColumn('paid_amount',f.regexp_replace(f.col('paid_amount'),',','.'))
        .withColumn('paid_amount',f.regexp_replace(f.col('paid_amount'),r'\s+',''))
        .withColumn('paid_amount',f.expr('try_cast(paid_amount as double)'))
        .withColumn('processed_at',f.to_timestamp('processed_at'))
        .withColumn('rank',f.row_number().over(window_df))
        .filter(f.col('rank')==1)
        .drop('rank')
        .withColumn('silver_run_id',f.lit(silver_run_id)))

    upsert_to_silver(payments_df_cleaned,'ecommerce_data.silver.payments_cleaned','payment_id')

    payments_validated=(
        payments_df_cleaned
        .withColumn('to_be_verified_by_payments_team',
        f.when(f.col('order_id').isNull(),'Verify_order_id')
        .when(f.col('payment_status').isNull(),'Verify_payment_status')
        .when(f.col('paid_amount').isNull() | (f.col('paid_amount')<=0),'Verify_paid_amount')
        .otherwise('No Issues'))
        .withColumn('check_paid_amount',
                    f.when((f.col('paid_amount')<=0) | f.col('paid_amount').isNull(),f.lit(False))
                    .otherwise(f.lit(True)))
    )

    payments_good=payments_validated.filter((f.col('to_be_verified_by_payments_team')=='No Issues') &
    (f.col('check_paid_amount')==True))

    payments_bad=(
        payments_validated
        .filter((f.col('to_be_verified_by_payments_team')!='No Issues'))
        .withColumn('quarantine_ts',f.current_timestamp())
    )

    upsert_to_silver(payments_good,'ecommerce_data.silver.payments_transformed','payment_id')

    payments_bad.write.format('delta').mode('append').saveAsTable('ecommerce_data.silver.payments_quarantine')

    mx_ingested=payments_df.agg(f.max('bronze_ingested_at').alias('mx')).collect()[0]['mx']

    mx_run=(
        payments_df.filter(f.col('bronze_ingested_at')==f.lit(mx_ingested)).agg(f.max('bronze_run_id').alias('mx')).collect()[0]['mx']
    )

    upsert_silver_control('payments',mx_run,mx_ingested,products_good.count())

else:
    print("No new records to process from Bronze table")  

    upsert_silver_control('payments',None,last_payment_processed_at,payments_df_count)    

In [0]:
print('orders_transformed_count',spark.sql('select count(*) from ecommerce_data.silver.orders_cleaned').collect()[0][0])
print('products_transformed_count',spark.sql('select count(*) from ecommerce_data.silver.products_cleaned').collect()[0][0])
print('payments_transformed_count',spark.sql('select count(*) from ecommerce_data.silver.payments_cleaned').collect()[0][0])
display(spark.sql('select * from ecommerce_data.silver.silver_control'))


In [0]:
%sql
select * from ecommerce_data.silver.products_transformed